In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

In [3]:
# 强制更新 langchain-core 到 0.3.0 以上版本，并更新 langchain-openai
# !{sys.executable} -m pip install -U langchain-core langchain-openai

In [3]:
# from poliprompt import TextClassifier,MultiModalClassifier
from poliprompt import MultiModalClassifier

2026-02-22 07:34:50,141 - faiss.loader - INFO - Loading faiss with AVX2 support.
2026-02-22 07:34:50,163 - faiss.loader - INFO - Successfully loaded faiss with AVX2 support.
d:\PoliPrompt-main\poliprompt_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# data_file = "../examples/topic_data.csv"
data_file = "../examples/HatefulMemes/train.jsonl"
work_station = "../examples/TopicExperiment"
env_path = "../.env"
options = ["1", "0"]
image_dir="../examples/HatefulMemes"
feature_col = "text"
answer_col = "label"
image_col="img"
random_state = 42
requests_per_period = 60
seconds_per_period = 60

In [5]:
import logging

# 配置日志格式
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
# 在 Notebook 环境里创建一个 logger
logger = logging.getLogger("Notebook")

In [6]:
topic_classifier = MultiModalClassifier(data_file, work_station, env_path, options, image_col=image_col, image_dir=image_dir)
logger.info("Running in Multi-Modal Mode.")

2026-02-22 07:35:03,329 - Notebook - INFO - Running in Multi-Modal Mode.


In [7]:
embedding_llm_name = "qwen"
reduce_method = "umap"
select_method = "kmeans"
testing = True
testing_size = 512
kshots = 5
lambda_param = 1.0

In [8]:
topic_classifier.create_few_shot_pool(embedding_llm_name, reduce_method, select_method, testing, testing_size)

2026-02-22 07:35:03,903 - poliprompt.multimodal_classifier - WARNING - You are supposed to provide 'embedding_llm_configs.json', 'reduce_configs.json', and 'select_configs.json' in the path ..\examples\TopicExperiment\infiles\configs
d:\PoliPrompt-main\poliprompt_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-02-22 07:53:08,166 - poliprompt.multimodal_classifier - INFO - NumPy array saved to ..\examples\TopicExperiment\outfiles\embeddings.index
2026-02-22 07:53:09,948 - poliprompt.multimodal_classifier - INFO - List of integer indices saved to ..\examples\TopicExperiment\outfiles\exemplar_indices.json
2026-02-22 07:53:09,951 - poliprompt.multimodal_classifier - INFO - Computation time: 00:18:06
2026-02-22 07:53:09,951 - poliprompt.multimodal_classifier - INFO - You are supposed to provid label in ..\examples\HatefulMemes\train.jsonl for the next steps.


In [9]:
llm_name = "gpt-4o-mini"
prompt_file_name = "hateful_v1.txt"

In [10]:
enhanced_description = topic_classifier.optimize_task_description(llm_name, prompt_file_name)

--- PHASE 2 (REDUCE): Synthesizing Global Knowledge ---


2026-02-22 07:53:21,381 - poliprompt.multimodal_classifier - INFO - Map-Reduce Optimization completed successfully.


In [11]:
enhanced_description

'### Concise Summary Rules for Meme Classification\n\n1. **Humor and Exaggeration**: Content that uses humor, satire, or exaggeration without promoting actual violence or hate is classified as **Not Hateful/Harmful (0)**. \n   - Example: Jokes about pets or light-hearted situations.\n\n2. **Trivialization of Violence or Tragedy**: Content that makes light of serious topics, such as violence or historical tragedies, especially in a derogatory manner, is classified as **Hateful/Harmful (1)**.\n   - Example: References to terrorism or serious crimes in a trivializing context.\n\n3. **Promotion of Harmful Behavior**: Any content that explicitly promotes or trivializes harmful behaviors, such as sexual violence or discrimination, is classified as **Hateful/Harmful (1)**.\n   - Example: References to pedophilia or other serious crimes.\n\n4. **Relatable Sentiments**: Content that expresses relatable emotions or experiences without negative implications is classified as **Not Hateful/Harmful 

In [16]:
llm_a_name = "gpt-4o-mini"
llm_b_name = "gemini-1.5-flash-latest"
# llm_b_name = "claude-3-haiku-20240307"
# llm_b_name = "gpt-4o-mini"
llm_adv_name = "gpt-4o"

[autoreload of poliprompt.multimodal_classifier failed: Traceback (most recent call last):
  File "d:\PoliPrompt-main\poliprompt_env\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\PoliPrompt-main\poliprompt_env\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 546, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
UnicodeDecodeError: 'gbk' codec can't decode byte 0x80 in position 202: illegal multibyte sequence
]


In [25]:
# test example
topic_classifier.annotate(llm_a_name, llm_b_name,llm_adv_name,prompt_file_name, kshots, lambda_param, testing, testing_size)

DEBUG: Testing mode active. Exemplar pool filtered to 60 items.
--- Starting Agentic Annotation (Total: 512) ---
DEBUG: query_embedding 形状: (1, 24)
DEBUG: pool_embeddings 形状: (60, 24)
--- [Row 61] Layer 2 (Model B) Cross-checking ---
Error: 'NoneType' object has no attribute 'label'


In [24]:
# --- 紧急手动收割脚本 (修正版) ---
import pandas as pd

# 1. 手动重建 config 对象
# 这里的 dataset_name 要和你 annotate 里的逻辑一致
dataset_name = topic_classifier.data_file.stem 
config = {"configurable": {"thread_id": f"job_{dataset_name}"}}

# 2. 从 agent_app 中抓取内存数据
try:
    snapshot = topic_classifier.agent_app.get_state(config)
    current_results = snapshot.values.get("results", {})

    # 3. 转换成 CSV
    harvest_df = topic_classifier.df.copy()
    for str_idx, res in current_results.items():
        idx = int(str_idx)
        # 确保不会因为索引问题崩掉
        if idx < len(harvest_df):
            harvest_df.at[idx, "predicted_label"] = res["label"]
            harvest_df.at[idx, "inference_path"] = res["path"]
            harvest_df.at[idx, "acc_cost_usd"] = res["cost"]

    # 4. 保存到一个确定的位置
    save_path = "../examples/TopicExperiment/outfiles/v3_salvage_61_samples.csv"
    harvest_df.to_csv(save_path, index=False)
    
    print(f"✅ 成功收割了 {len(current_results)} 条 v3.0 异构仲裁的结果！")
    print(f"文件已保存至: {save_path}")
    
    # 5. 顺手算出路径分布，直接给 PPT 用
    if current_results:
        path_series = pd.Series([res['path'] for res in current_results.values()])
        print("\n📊 PPT 直接可用数据 (Path Distribution):")
        print(path_series.value_counts(normalize=True) * 100)

except Exception as e:
    print(f"❌ 收割失败，错误原因: {e}")

✅ 成功收割了 61 条 v3.0 异构仲裁的结果！
文件已保存至: ../examples/TopicExperiment/outfiles/v3_salvage_61_samples.csv

📊 PPT 直接可用数据 (Path Distribution):
L2_Heterogeneous_Match    81.967213
L3_Expert_Consensus       18.032787
Name: proportion, dtype: float64


In [27]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. 加载收割到的 61 个样本
file_path = "../examples/TopicExperiment/outfiles/v3_salvage_61_samples.csv"
df = pd.read_csv(file_path)

# 2. 数据清洗：只评估已经有预测结果的行
eval_df = df.dropna(subset=['predicted_label']).copy()
eval_df['label'] = eval_df['label'].astype(int)
eval_df['predicted_label'] = eval_df['predicted_label'].astype(int)

# 3. 计算核心指标
acc = accuracy_score(eval_df['label'], eval_df['predicted_label'])
f1 = f1_score(eval_df['label'], eval_df['predicted_label'], average='macro')

print(f"\n--- 📊 PoliPrompt v3.0 Quantitative Results ---")
print(f"Sample Size (N): {len(eval_df)}")
print(f"Accuracy: {acc:.2%}")
print(f"Macro-F1 Score: {f1:.4f}")

# 4. 路径分布分析 (这是你 PPT Slide 3 的核心数据)
print("\n--- 🛣 Inference Path Distribution (PPT Highlight) ---")
path_counts = eval_df['inference_path'].value_counts()
for path, count in path_counts.items():
    percentage = (count / len(eval_df)) * 100
    print(f"{path}: {percentage:.1f}% ({count} samples)")

# 5. 成本与效率分析 (PPT Slide 3 成本部分)
if 'acc_cost_usd' in eval_df.columns:
    total_cost = eval_df['acc_cost_usd'].iloc[-1]
    avg_cost = total_cost / len(eval_df)
    print("\n--- 💰 Cost Efficiency ---")
    print(f"Total API Cost: ${total_cost:.4f}")
    print(f"Avg Cost per Sample: ${avg_cost:.4f}")
    
# 6. 查找纠偏成功的案例
# 找 A!=B (走过 L3) 且最后做对了的
correction = eval_df[eval_df['inference_path'].str.contains('L3', na=False) & (eval_df['label'] == eval_df['predicted_label'])]
print(f"\n✨ Successful L3 Arbitrations: {len(correction)}")
if not correction.empty:
    print("Example Row ID for Case Study:", correction['id'].iloc[0])


--- 📊 PoliPrompt v3.0 Quantitative Results ---
Sample Size (N): 61
Accuracy: 77.05%
Macro-F1 Score: 0.6369

--- 🛣 Inference Path Distribution (PPT Highlight) ---
L2_Heterogeneous_Match: 82.0% (50 samples)
L3_Expert_Consensus: 18.0% (11 samples)

--- 💰 Cost Efficiency ---
Total API Cost: $0.6525
Avg Cost per Sample: $0.0107

✨ Successful L3 Arbitrations: 7
Example Row ID for Case Study: 70914
